# **HW7 – Quantum Feature Selection, Neural Networks, and Hybrid Methods**
_Time required: ~2–3 hours (for students with Qiskit ML and kernel experience)_

**What you’ll practice**
- Quantum mutual information for feature ranking
- Building and training variational quantum neural networks
- Designing hybrid quantum-classical architectures
- Integrating quantum circuits with classical optimizers
- Evaluating hybrid vs pure models on classification tasks
- Noise impact on QNN training and selection

**What to turn in**
- This single notebook (`HW7_YourName.ipynb`) with **all cells run**, code and short written answers filled in where prompted.

**Rules & hints**
- Use **Qiskit** (version ~2.0 or later), **Qiskit Machine Learning**.
- Use AerSimulator for reproducibility.
- Focus on core tasks; skip extras if time-constrained.


In [2]:
# !python -m pip install --upgrade pip
# !pip uninstall qiskit qiskit-aer qiskit-ibm-runtime qiskit-machine-learning matplotlib sklearn -y
!pip install qiskit qiskit-aer qiskit-ibm-runtime qiskit-machine-learning qiskit_algorithms matplotlib


  Using cached qiskit_aer-0.17.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.3 kB)
  Using cached qiskit_ibm_runtime-0.43.0-py3-none-any.whl.metadata (21 kB)
  Using cached qiskit_machine_learning-0.8.4-py3-none-any.whl.metadata (13 kB)
  Using cached matplotlib-3.10.7-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached qiskit-1.4.5-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
Using cached qiskit_aer-0.17.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (12.4 MB)
Using cached qiskit_ibm_runtime-0.43.0-py3-none-any.whl (1.4 MB)
Using cached qiskit_machine_learning-0.8.4-py3-none-any.whl (231 kB)
Using cached qiskit-1.4.5-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (6.8 MB)
Using cached matplotlib-3.10.7-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (8.7 MB)
  Attempting uninstall: qiskit
    Found existing installation: qiskit 2.2.1
    Uninstalling qiskit

In [3]:
!pip install pylatexenc
import pylatexenc
# --- Setup (run me first) ---
import sklearn
# Import necessary modules
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import DensityMatrix, partial_trace
from qiskit.circuit.library import ZZFeatureMap, TwoLocal
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms import VQC
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.connectors import TorchConnector
# from qiskit_algorithms.optimizers import SPSA
from scipy.optimize import minimize
from sklearn import datasets, metrics
from sklearn.neural_network import MLPClassifier
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn, optim
from qiskit.circuit import QuantumCircuit, Parameter
from qiskit.circuit.library import zz_feature_map
from qiskit.quantum_info import mutual_information, Statevector
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer import AerSimulator
from qiskit import transpile
from IPython.display import display, HTML, Image
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score

import copy
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import z_feature_map  # modern helper
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator as Estimator  # V2 simulator


# --- utilities: split circuit parameters and build bindings ---
from typing import List, Tuple
from qiskit.circuit import Parameter
# Simulator backend
sim = AerSimulator()

# Function to run circuit and get counts
def get_counts(circ, shots=1024):
    tc = transpile(circ, sim)
    result = sim.run(tc, shots=shots).result()
    return result.get_counts()

def assert_close(A, B, tol=1e-4):
    if not np.allclose(A, B, atol=tol):
        raise AssertionError(f"Not close: {A} vs {B}")


## Part A — Quantum Feature Selection (Sessions 11) (≈45 min)

**A1.** I've implemented a swap-test circuit to compute fidelity between two 1-qubit feature states (use ZZFeatureMap for dim=1). Test on x1=0.5, x2= -0.5.  
**A2.**I've computed QMI for a toy feature-label pair; rank 3 iris features by avg QMI with labels.  
**A3.** Your turn.  Expand this to the full Iris Dataset


In [ ]:
# A1. Swap test fidelity


data = [0.1, 0.2, 0.3]

zz_feature_map_reference = zz_feature_map(feature_dimension=3, reps=2)
zz_feature_map_reference = zz_feature_map_reference.assign_parameters(data)
zz_feature_map_reference.decompose().draw("mpl")
qc_swap = QuantumCircuit(3, 1)
# YOUR CODE HERE: compose qc1 on 1, qc2 on 2; h(0), cs(0,1,2), h(0), measure 0
qc_swap.h(0)
qc_swap.cswap(0,1,2)
qc_swap.h(0)
qc = zz_feature_map_reference.compose(qc_swap)
qc.measure_all()
qc.draw('mpl')
plt.show()
counts = get_counts(qc)
fid = 2 * (counts.get('0', 0) / sum(counts.values())) - 1
print("Fidelity: ", fid)

# A2. QMI ranking on iris
# Data
iris = datasets.load_iris()
X = iris.data[:, :3]           # 3 features
X = (X - X.min(0)) / (X.max(0) - X.min(0) + 1e-12)  # normalize to [0,1]
y = iris.target                # classes {0,1,2}

def encode_label_two_qubits(qc, y_val, q0, q1):
    # map 0->00, 1->01, 2->10 (leave 3 unused)
    if y_val == 1:
        qc.x(q1)
    elif y_val == 2:
        qc.x(q0)

def empirical_density_matrix(feature_col, labels):
    """Build ρ_AB with A=1 feature qubit, B=2 label qubits (dims=(2,4))."""
    rho = None
    for xi, yi in zip(feature_col, labels):
        qc = QuantumCircuit(3)  # q0=A (feature), q1,q2=B (label)
        # angle encoding (simple and common): map x∈[0,1] -> Ry(π x)
        qc.ry((np.pi * xi), [0])
        encode_label_two_qubits(qc, yi, 1, 2)
        psi = Statevector.from_instruction(qc)
        rho_i = DensityMatrix(psi)
        rho = rho_i if rho is None else (rho + rho_i)
    rho = rho / len(feature_col)
    # Make sure dims reflect the bipartition (A=2, B=4)
    rho = DensityMatrix(rho.data, dims=(2, 4))
    return rho

# QMI for each of the first 3 features vs. the 2-qubit label register
qmi_scores = []
for j in range(3):
    rho = empirical_density_matrix(X[:, j], y)
    qmi_scores.append(mutual_information(rho, base=2))

print("QMI scores (feature 0..2):", qmi_scores)

# A3. Written answer: (QMI detects entanglement; hybrid: quantum scores + classical greedy/ranking)


In [ ]:
# A1. Swap test fidelity (Qiskit 2.x, 3 features from Iris)


# --- data ---
iris = datasets.load_iris()
X_all = iris.data[:, :3]              # 3 features
x_vec  = X_all[0]                     # sample 1 (bind here)
x_ref  = X_all[1]                     # sample 2 (reference)
n = 3

# --- parameterized feature map with explicit ParameterVector ---
x = ParameterVector('x', n)
# --- parameterized feature map (let it create its own 'x' params) ---
fm = ZZFeatureMap(feature_dimension=n, reps=2, parameter_prefix='x')

# ordered parameters from the feature map
fm_params = sorted(fm.parameters, key=lambda p: p.name)

# bind parameters to two copies
fm_x   = fm.assign_parameters({p: v for p, v in zip(fm_params, x_vec)})
fm_ref = fm.assign_parameters({p: v for p, v in zip(fm_params, x_ref)})

# --- swap test: 1 ancilla + n + n qubits ---
qc = QuantumCircuit(1 + 2*n, 1)
qc.compose(fm_x,   qubits=range(1, n+1), inplace=True)         # |φ(x)⟩ on 1..n
qc.compose(fm_ref, qubits=range(n+1, 2*n+1), inplace=True)     # |φ(ref)⟩ on n+1..2n

qc.h(0)
for i in range(n):
    qc.cswap(0, 1+i, 1+n+i)
qc.h(0)
qc.measure(0, 0)

# simulate
sim = AerSimulator()
job = sim.run(transpile(qc, sim), shots=4096)
counts = job.result().get_counts()

p0  = counts.get('0', 0) / sum(counts.values())
fid = 2*p0 - 1                       # ≈ |⟨φ(x)|φ(ref)⟩|^2 for swap test
print("Fidelity:", fid)

# optional draw
plot = qc.draw('mpl')
display(plot)

## Part B — Quantum Neural Networks (Session 12) (≈45 min)

**B1.** I built a VQC for moons classification (2 qubits, reps=1); train with SPSA (maxiter=50).  
**B2.** I evaluate accuracy; compare to classical MLP (hidden=4).  
**B3.** Your turn.  Try to increase the accuracy of these quantum methods.


In [ ]:
# B1. VQC on moons
X_m, y_m = datasets.make_moons(n_samples=100, noise=0.1)
feature_map = ZZFeatureMap(2)
ansatz = TwoLocal(2, 'ry', 'cz', reps=1)
vqc = VQC(feature_map=feature_map, ansatz=ansatz, optimizer=minimize)
vqc.fit(X_m, y_m)

# B2. Acc compare
acc_vqc = vqc.score(X_m, y_m)
mlp = MLPClassifier(hidden_layer_sizes=(4,), max_iter=200)
mlp.fit(X_m, y_m)
acc_mlp = mlp.score(X_m, y_m)
print("VQC acc: ", acc_vqc, " MLP: ", acc_mlp)



In [7]:
# Modern Qiskit VQC-style classifier (EstimatorV2, no qiskit-ml)
# Pattern: encode with z_feature_map → variational ansatz → Estimator(V2) expectation of Z…Z → MSE loss


def split_params(circ) -> Tuple[List[Parameter], List[Parameter], List[Parameter]]:
    """
    Returns (all_params_sorted, data_params, theta_params).
    We treat everything named 'θ[...]' as trainable weights; the rest are data.
    This works whether your circuit has one or many data prefixes (a, a0, a1, ...).
    """
    all_params = sorted(list(circ.parameters), key=lambda p: p.name)
    theta_params = [p for p in all_params if p.name.startswith("θ[")]
    data_params  = [p for p in all_params if p not in theta_params]
    return all_params, data_params, theta_params

def make_bindings(circ, X_batch, w_vec):
    """
    Returns a dict binding matrix keyed by the parameter tuple expected by EstimatorV2.
    Shapes:
      X_batch: (B, #data_params)
      w_vec:   (#theta_params,)
    """
    all_params, data_params, theta_params = split_params(circ)
    B = X_batch.shape[0]
    assert X_batch.shape[1] == len(data_params), \
        f"X has {X_batch.shape[1]} cols but circuit needs {len(data_params)} data params."
    assert len(w_vec) == len(theta_params), \
        f"w has {len(w_vec)} but circuit has {len(theta_params)} θ-params."

    W = np.broadcast_to(np.asarray(w_vec), (B, len(theta_params)))
    # Rebuild columns in the *exact* order of all_params
    # (data first or not — doesn't matter now, because we place columns by name)
    col_map = {p: None for p in all_params}
    # fill data columns
    for j, p in enumerate(data_params):
        col_map[p] = X_batch[:, j]
    # fill theta columns
    for j, p in enumerate(theta_params):
        col_map[p] = W[:, j]
    # stack columns according to all_params
    mat = np.stack([col_map[p] for p in all_params], axis=1)  # (B, len(all_params))
    return {tuple(all_params): mat}

# ----------------------------
# Data (2D moons) → scale to [0, π]
# ----------------------------
X, y01 = datasets.make_moons(n_samples=300, noise=0.15, random_state=7)
scaler = MinMaxScaler(feature_range=(0.0, np.pi))
X = scaler.fit_transform(X)

# Map labels {0,1} → {-1,+1} to match Z-eigenvalues
y = np.where(y01 == 1, 1.0, -1.0)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=7)
num_qubits = X.shape[1]  # 2

# ----------------------------
# Circuit: feature map + lightweight ansatz
# ----------------------------
# Low-depth data encoding
feature_map = z_feature_map(num_qubits, parameter_prefix="a")

# Simple variational block: Ry layer → CZ → Rz layer
params = ParameterVector("θ", length=2 * num_qubits)
# ansatz = QuantumCircuit(num_qubits)
# for q in range(num_qubits):
#     ansatz.ry(params[q], q)
# # entangle adjacent (linear) with CZs
# for q in range(num_qubits - 1):
#     ansatz.cz(q, q + 1)
# for q in range(num_qubits):
#     ansatz.rz(params[num_qubits + q], q)
def make_deep_ansatz(nq: int, L: int = 3) -> QuantumCircuit:
    thetas = ParameterVector("θ", length=2*nq*L)
    qc = QuantumCircuit(nq)
    k = 0
    for _ in range(L):
        for q in range(nq):           # local rotations
            qc.ry(thetas[k], q); k += 1
        # grid entanglement (assumes rows×cols layout)
        rows = int(np.sqrt(nq)); cols = nq // rows
        for r in range(rows):
            for c in range(cols-1):   # horizontal CZs
                qc.cz(r*cols+c, r*cols+c+1)
        for r in range(rows-1):
            for c in range(cols):     # vertical CZs
                qc.cz(r*cols+c, (r+1)*cols+c)
        for q in range(nq):
            qc.rz(thetas[k], q); k += 1
    return qc

ansatz = make_deep_ansatz(num_qubits, L=3)  # increase L to raise expressivity


# Full circuit = encode then ansatz
full_circuit = QuantumCircuit(num_qubits)
full_circuit.compose(feature_map, range(num_qubits), inplace=True)
full_circuit.compose(ansatz, range(num_qubits), inplace=True)

# Observable: product Z…Z → expectation in [-1, +1]
observable = SparsePauliOp.from_list([("Z" * num_qubits, 1.0)])

# ----------------------------
# Forward pass (EstimatorV2)
# ----------------------------
def forward(circuit: QuantumCircuit,
            input_params: np.ndarray,
            weight_params: np.ndarray,
            estimator: Estimator,
            observable: SparsePauliOp) -> np.ndarray:
    """
    Returns one expectation per sample for the given observable.
    """
    # Broadcast weights over samples and concatenate after input params.
    # Parameter order in 'full_circuit' is alphabetical; we used prefix "a" for features,
    # so those come first, then the ansatz thetas "θ".
    num_samples = input_params.shape[0]
    W = np.broadcast_to(weight_params, (num_samples, len(weight_params)))
    bound_params = np.concatenate([input_params, W], axis=1)

    # Single publication with batched param sets
    pub = (circuit, observable, bound_params)
    result = estimator.run([pub]).result()[0]
    return np.asarray(result.data.evs)

# ----------------------------
# Loss (MSE) and objective for optimizer
# ----------------------------
def mse_loss(pred: np.ndarray, target: np.ndarray) -> float:
    return float(((pred - target) ** 2).mean())

# Globals used by the objective (mirrors the lesson’s structure)
estimator = Estimator()
circuit = full_circuit
obs = observable
input_params = X_tr
target = y_tr
objective_vals = []
_iter = {"k": 0}  # tiny mutable counter

def mse_loss_weights(w: np.ndarray) -> float:
    preds = forward(circuit, input_params, w, estimator, obs)
    cost = mse_loss(preds, target)
    objective_vals.append(cost)
    _iter["k"] += 1
    if _iter["k"] % 50 == 0:
        print(f"iter={_iter['k']}, loss={cost:.6f}")
    return cost

# ----------------------------
# Train
# ----------------------------
np.random.seed(42)
# --- forward (EstimatorV2, explicit bindings) ---
def forward(circuit, input_params, weight_params, estimator, observable):
    bindings = make_bindings(circuit, input_params, weight_params)
    pub = (circuit, observable, bindings)
    res = estimator.run([pub]).result()[0]
    return np.asarray(res.data.evs)

# --- initialize weights directly from the circuit (not from a stale 'params' vector) ---
all_params, data_params, theta_params = split_params(full_circuit)
np.random.seed(42)
w0 = np.random.uniform(0, 2*np.pi, size=len(theta_params))

# --- train ---
objective_vals = []
_iter = {"k": 0}

def mse_loss(pred, target): return float(((pred - target)**2).mean())

def mse_loss_weights(w):
    preds = forward(full_circuit, X_tr, w, estimator, observable)
    cost = mse_loss(preds, y_tr)
    objective_vals.append(cost)
    _iter["k"] += 1
    if _iter["k"] % 50 == 0:
        print(f"iter={_iter['k']}, loss={cost:.6f}")
    return cost

res = minimize(mse_loss_weights, w0, method="COBYQA", options={"maxiter": 150})
w_opt = res.x


# ----------------------------
# Evaluate
# ----------------------------
def predict_labels(Xdata: np.ndarray, w: np.ndarray) -> np.ndarray:
    preds = forward(full_circuit, Xdata, w, estimator, observable)
    labels = np.where(preds >= 0.0, 1.0, -1.0)
    return labels, preds

yhat_tr, pred_tr = predict_labels(X_tr, w_opt)
yhat_te, pred_te = predict_labels(X_te, w_opt)

acc_tr = accuracy_score(y_tr, yhat_tr)
acc_te = accuracy_score(y_te, yhat_te)

print(f"Train accuracy: {acc_tr:.3f}")
print(f"Test  accuracy: {acc_te:.3f}")
# Optional: inspect optimization trace -> 'objective_vals'


iter=50, loss=0.682790
iter=100, loss=0.646280
iter=150, loss=0.644466
iter=200, loss=0.643944
Train accuracy: 0.800
Test  accuracy: 0.767


In [8]:
# Classical baselines on the same data/split


# y is in {-1,+1}; most sklearn classifiers expect {0,1}
y_tr01 = (y_tr > 0).astype(int)
y_te01 = (y_te > 0).astype(int)

logreg = LogisticRegression(max_iter=2000, solver="lbfgs")
logreg.fit(X_tr, y_tr01)
acc_lr_tr = accuracy_score(y_tr01, logreg.predict(X_tr))
acc_lr_te = accuracy_score(y_te01, logreg.predict(X_te))
print(f"LogReg accuracy — train: {acc_lr_tr:.3f}, test: {acc_lr_te:.3f}")

svm = SVC(kernel="rbf", C=1.0, gamma="scale")
svm.fit(X_tr, y_tr01)
acc_svm_tr = accuracy_score(y_tr01, svm.predict(X_tr))
acc_svm_te = accuracy_score(y_te01, svm.predict(X_te))
print(f"SVM(RBF) acc   — train: {acc_svm_tr:.3f}, test: {acc_svm_te:.3f}")

mlp = MLPClassifier(hidden_layer_sizes=(16,), activation="relu",
                    max_iter=2000, random_state=7)
mlp.fit(X_tr, y_tr01)
acc_mlp_tr = accuracy_score(y_tr01, mlp.predict(X_tr))
acc_mlp_te = accuracy_score(y_te01, mlp.predict(X_te))
print(f"MLP acc        — train: {acc_mlp_tr:.3f}, test: {acc_mlp_te:.3f}")

# Compare with your quantum model (already computed)
print(f"VQC-style acc  — train: {acc_tr:.3f}, test: {acc_te:.3f}")


LogReg accuracy — train: 0.848, test: 0.922
SVM(RBF) acc   — train: 0.976, test: 0.978
MLP acc        — train: 0.829, test: 0.867
VQC-style acc  — train: 0.800, test: 0.767
